# 01 — Data Exploration

This notebook:
1. Downloads nflfastR play-by-play data (or loads from cache).
2. Generates synthetic in-play odds.
3. Builds game-state and odds-path features.
4. Produces exploratory visualizations.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('inline')
import yaml

# Load config
with open('../config/default.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded. Seasons:', cfg['data']['seasons'])

## 1.1 Download / Load Play-by-Play Data

We use `nfl_data_py` to fetch nflfastR PBP.  On first run this downloads ~100 MB; subsequent runs load from the parquet cache.

In [ ]:
from src.data_ingestion.play_by_play import load_pbp

# Use a single season for exploration (faster)
SEASONS = [2022]

pbp = load_pbp(seasons=SEASONS, cache_dir='../data/processed')
print(f'Loaded {len(pbp):,} plays across {pbp["game_id"].nunique():,} games')
print(pbp.dtypes)

In [ ]:
pbp.head()

## 1.2 Win Probability Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
pbp['wp'].dropna().hist(bins=50, ax=ax, color='#1f77b4', edgecolor='white')
ax.set_xlabel('nflfastR Win Probability')
ax.set_ylabel('Count')
ax.set_title('Distribution of Play-Level WP')

ax = axes[1]
pbp['epa'].dropna().clip(-5, 5).hist(bins=60, ax=ax, color='#ff7f0e', edgecolor='white')
ax.set_xlabel('Expected Points Added (EPA)')
ax.set_ylabel('Count')
ax.set_title('Distribution of EPA')

plt.tight_layout()
plt.show()

## 1.3 Single-Game WP Path

In [ ]:
# Pick the first game in the dataset
sample_gid = pbp['game_id'].unique()[0]
game_pbp = pbp[pbp['game_id'] == sample_gid].copy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(game_pbp['elapsed_seconds'], game_pbp['wp'], color='#1f77b4', lw=1.5)
ax.axhline(0.5, color='grey', ls='--', lw=0.8)
ax.set_xlabel('Elapsed Game Seconds')
ax.set_ylabel('Win Probability (home/possession team)')
ax.set_title(f'WP Path — Game {sample_gid}')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 1.4 Generate Synthetic Odds

In [ ]:
from src.data_ingestion.odds_simulator import SyntheticOddsGenerator

sim_cfg = cfg['odds_simulator']
generator = SyntheticOddsGenerator(
    sigma_micro=sim_cfg['sigma_micro'],
    momentum_factor_min=sim_cfg['momentum_factor_min'],
    momentum_factor_max=sim_cfg['momentum_factor_max'],
    base_volume_lambda=sim_cfg['base_volume_lambda'],
    spike_volume_lambda=sim_cfg['spike_volume_lambda'],
    seed=42,
)

# Generate for a single game
game_odds = generator.generate(game_pbp)
print(game_odds.head())
print(f'\nOdds generated: {len(game_odds)} rows')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax = axes[0]
ax.plot(game_odds['elapsed_seconds'], game_odds['true_wp'],
        label='True WP (nflfastR)', lw=1.5, color='#1f77b4')
ax.plot(game_odds['elapsed_seconds'], game_odds['book_implied_prob'],
        label='Book Implied Prob', lw=1.0, ls='--', alpha=0.8, color='#2ca02c')
ax.set_ylabel('Probability')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.set_title('True WP vs Book Implied Probability')

ax = axes[1]
ax.bar(game_odds['elapsed_seconds'], game_odds['volume'],
       width=30, color='#9467bd', alpha=0.6)
ax.set_xlabel('Elapsed Seconds')
ax.set_ylabel('Volume')
ax.set_title('Synthetic Betting Volume')

plt.tight_layout()
plt.show()

## 1.5 Feature Engineering

In [ ]:
from src.feature_engineering.game_state_features import build_game_state_features
from src.feature_engineering.odds_path_features import build_odds_path_features

game_pbp_feat = build_game_state_features(game_pbp)
game_odds_feat = build_odds_path_features(game_odds)

print('Game-state features added:')
new_cols = ['down_distance_bucket', 'field_position_bucket', 'score_diff_bucket',
            'quarter', 'is_two_minute_warning', 'rolling_epa_5',
            'rolling_success_rate_10', 'drive_play_count']
print(game_pbp_feat[new_cols].head(10))

In [ ]:
print('Odds-path features added:')
odds_cols = ['elapsed_seconds', 'book_implied_prob', 'odds_return',
             'rolling_odds_vol_10', 'odds_momentum_5', 'volume_surge', 'price_impact']
print(game_odds_feat[odds_cols].head(10))

In [ ]:
# Volume surge fraction
surge_rate = game_odds_feat['volume_surge'].mean()
print(f'Volume surge rate: {surge_rate:.1%} of plays')

## 1.6 Field Position vs Scoring Rate

In [ ]:
# Use the full single-season PBP for this analysis
pbp_feat = build_game_state_features(pbp)
pbp_feat['is_scoring'] = (pbp_feat['touchdown'].fillna(0) > 0).astype(int)

score_by_pos = (
    pbp_feat.groupby('field_position_bucket', observed=True)['is_scoring']
    .agg(['mean', 'sum', 'count'])
    .rename(columns={'mean': 'scoring_rate', 'sum': 'scoring_plays', 'count': 'total_plays'})
)

fig, ax = plt.subplots(figsize=(8, 4))
order = ['own_territory', 'midfield', 'opponent_territory', 'red_zone']
valid = [o for o in order if o in score_by_pos.index]
bars = score_by_pos.loc[valid, 'scoring_rate']
ax.bar(valid, bars * 100, color=['#1f77b4', '#aec7e8', '#ffbb78', '#d62728'])
ax.set_xlabel('Field Position')
ax.set_ylabel('Touchdown Rate (%)')
ax.set_title('Touchdown Rate by Field Position')
plt.tight_layout()
plt.show()

print(score_by_pos)

---
**Next:** Open `02_model_fitting.ipynb` to fit the Hawkes process, Kalman filter, and scoring intensity model.